In [0]:
import random
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from datetime import datetime, timedelta

# 1. Define the schema for our financial transaction data
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("trade_date", TimestampType(), False),
    StructField("security_isin", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("volume", IntegerType(), True),
    StructField("trader_pan", StringType(), True)
])

# 2. Helper lists to generate realistic data
isins = ["INE406A01037", "INE002A01018", "INE040A01034", None] # Includes a null to test data quality rules
pans = ["ABCDE1234F", "XYZWX5678G", "LMNOP9012H", "JKXYZ4321I"]

raw_data = []
base_time = datetime.now()

# 3. Generate 100 sample records
for i in range(100):
    tx_id = f"TXN-{10000 + i}"
    trade_time = base_time - timedelta(minutes=random.randint(1, 1440))
    isin = random.choice(isins)
    
    # Introduce an intentional data anomaly: negative price or negative volume
    price = round(random.uniform(150.0, 2500.0), 2) if i % 15 != 0 else -50.0 
    volume = random.randint(10, 5000) if i % 20 != 0 else -100
    
    pan = random.choice(pans)
    
    raw_data.append((tx_id, trade_time, isin, price, volume, pan))

# 4. Create the Spark DataFrame
df = spark.createDataFrame(raw_data, schema)

# 5. Save the data as a raw landing Delta table
# In Community Edition, we save directly to DBFS (Databricks File System)
df.write.format("delta").mode("overwrite").saveAsTable("raw_financial_transactions")

print("Successfully generated 100 financial transaction records and saved to 'raw_financial_transactions'!")

Successfully generated 100 financial transaction records and saved to 'raw_financial_transactions'!


In [0]:
%sql
SELECT * FROM raw_financial_transactions LIMIT 10;

transaction_id,trade_date,security_isin,price,volume,trader_pan
TXN-10000,2026-06-02T10:05:21.255Z,null,-50.0,-100,XYZWX5678G
TXN-10001,2026-06-02T04:45:21.255Z,INE406A01037,582.51,3173,JKXYZ4321I
TXN-10002,2026-06-02T12:37:21.255Z,null,1786.23,834,ABCDE1234F
TXN-10003,2026-06-02T07:00:21.255Z,INE040A01034,1194.06,1894,XYZWX5678G
TXN-10004,2026-06-02T12:25:21.255Z,INE040A01034,1146.2,1871,JKXYZ4321I
TXN-10005,2026-06-02T04:46:21.255Z,null,1387.7,3604,JKXYZ4321I
TXN-10006,2026-06-01T14:49:21.255Z,INE040A01034,1426.84,4055,LMNOP9012H
TXN-10007,2026-06-02T03:46:21.255Z,INE406A01037,2352.94,4765,JKXYZ4321I
TXN-10008,2026-06-01T23:57:21.255Z,INE002A01018,2084.07,120,LMNOP9012H
TXN-10009,2026-06-02T08:54:21.255Z,null,1751.36,2253,ABCDE1234F


In [0]:
%sql
SELECT 
    transaction_id,
    trade_date,
    security_isin,
    price,
    volume,
    trader_pan,
    -- Rule 1: Audit for Missing Security Identifiers
    CASE 
        WHEN security_isin IS NULL THEN 'FAILED_COMPLIANCE (Missing ISIN)'
        -- Rule 2: Audit for Operational Anomaly (Negative Prices/Volume)
        WHEN price <= 0 OR volume <= 0 THEN 'FAILED_AUDIT (Invalid Numeric Values)'
        ELSE 'PASSED'
    End AS data_quality_status
FROM raw_financial_transactions;

transaction_id,trade_date,security_isin,price,volume,trader_pan,data_quality_status
TXN-10000,2026-06-02T10:05:21.255Z,null,-50.0,-100,XYZWX5678G,FAILED_COMPLIANCE (Missing ISIN)
TXN-10001,2026-06-02T04:45:21.255Z,INE406A01037,582.51,3173,JKXYZ4321I,PASSED
TXN-10002,2026-06-02T12:37:21.255Z,null,1786.23,834,ABCDE1234F,FAILED_COMPLIANCE (Missing ISIN)
TXN-10003,2026-06-02T07:00:21.255Z,INE040A01034,1194.06,1894,XYZWX5678G,PASSED
TXN-10004,2026-06-02T12:25:21.255Z,INE040A01034,1146.2,1871,JKXYZ4321I,PASSED
TXN-10005,2026-06-02T04:46:21.255Z,null,1387.7,3604,JKXYZ4321I,FAILED_COMPLIANCE (Missing ISIN)
TXN-10006,2026-06-01T14:49:21.255Z,INE040A01034,1426.84,4055,LMNOP9012H,PASSED
TXN-10007,2026-06-02T03:46:21.255Z,INE406A01037,2352.94,4765,JKXYZ4321I,PASSED
TXN-10008,2026-06-01T23:57:21.255Z,INE002A01018,2084.07,120,LMNOP9012H,PASSED
TXN-10009,2026-06-02T08:54:21.255Z,null,1751.36,2253,ABCDE1234F,FAILED_COMPLIANCE (Missing ISIN)


In [0]:
%sql
CREATE OR REPLACE TABLE  default.financial_transactions_silver AS
SELECT 
    transaction_id,
    trade_date,
    security_isin,
    price,
    volume,
    trader_pan
FROM raw_financial_transactions
WHERE security_isin IS NOT NULL 
  AND price > 0 
  AND volume > 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE  default.compliance_quarantine_table AS
SELECT 
    transaction_id,
    trade_date,
    security_isin,
    price,
    volume,
    trader_pan,
    CASE 
        WHEN security_isin IS NULL THEN 'Missing Security ISIN'
        WHEN price <= 0 THEN 'Negative/Zero Trade Price'
        WHEN volume <= 0 THEN 'Negative/Zero Trade Volume'
    END AS rejection_reason
FROM raw_financial_transactions
WHERE security_isin IS NULL 
   OR price <= 0 
   OR volume <= 0;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'Clean Production Rows' as table_name, COUNT(*) as record_count FROM financial_transactions_silver
UNION ALL
SELECT 'Quarantined Audit Rows' as table_name, COUNT(*) as record_count FROM compliance_quarantine_table;

table_name,record_count
Clean Production Rows,66
Quarantined Audit Rows,34


In [0]:
%sql
CREATE OR REPLACE VIEW financial_transactions_gold AS
SELECT 
    transaction_id,
    trade_date,
    security_isin,
    price,
    volume,
    -- Senior Masking Logic: Hides the first 6 characters of the Indian PAN
    CONCAT('XXXXXX', RIGHT(trader_pan, 4)) AS masked_trader_pan
FROM financial_transactions_silver;

In [0]:
%sql
SELECT * FROM financial_transactions_gold LIMIT 10;

transaction_id,trade_date,security_isin,price,volume,masked_trader_pan
TXN-10001,2026-06-02T04:45:21.255Z,INE406A01037,582.51,3173,XXXXXX321I
TXN-10003,2026-06-02T07:00:21.255Z,INE040A01034,1194.06,1894,XXXXXX678G
TXN-10004,2026-06-02T12:25:21.255Z,INE040A01034,1146.2,1871,XXXXXX321I
TXN-10006,2026-06-01T14:49:21.255Z,INE040A01034,1426.84,4055,XXXXXX012H
TXN-10007,2026-06-02T03:46:21.255Z,INE406A01037,2352.94,4765,XXXXXX321I
TXN-10008,2026-06-01T23:57:21.255Z,INE002A01018,2084.07,120,XXXXXX012H
TXN-10012,2026-06-01T15:29:21.255Z,INE002A01018,1283.67,2472,XXXXXX321I
TXN-10013,2026-06-02T05:07:21.255Z,INE002A01018,1875.96,470,XXXXXX678G
TXN-10014,2026-06-02T10:41:21.255Z,INE002A01018,521.38,3708,XXXXXX321I
TXN-10016,2026-06-02T10:16:21.255Z,INE002A01018,579.25,612,XXXXXX012H
